# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariamsherif04/flyrank-ai/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

import duckdb, pandas as pd
from google.colab import userdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
DEV_MONTH = "2026-03"
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month={DEV_MONTH}/*.parquet')"
DIM_CLIENTS = f"read_parquet('{BASE}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{BASE}/dim_content.parquet')"

## 1. Unit of analysis + time window

One row in my lane = one content item × one day (content_hash_id × report_date), pulled from fact_content_daily_performance, joined to dim_content for static attributes like creation date. Time window: a mid-panel month, 2026-03, deliberately avoiding the final month (fact_content_daily_performance_sample, June 2026), which is the sealed test window — developing label logic there would mean testing on my own answer key. Verified below: row count, distinct content count, and date span all match a single clean month.

In [13]:
span_check = con.execute(f"""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT content_hash_id) AS distinct_content,
           MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {FACT}
""").df()
span_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_content,min_date,max_date
0,9841378,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

- Feature: gsc_impressions, gsc_clicks, gsc_avg_position, content_age_days (joined from dim_content.content_created_date) — all knowable before the moment I'd act on them.

- Label / proxy: whether a page's CTR ranks in the bottom half of this month's content (is_declining) — built fresh from ctr_raw, never taken from a pre-computed trend column.

- Context: content_hash_id, client_hash_id, report_date — used only for joining and grouping, never fed to a model.

- Excluded: any pre-aggregated trend/direction-style column. If the warehouse ever ships one, it would be computed from the same outcome I'm trying to predict, so it can never be a feature.

In [14]:
field_map = {
    "feature": ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "content_age_days (joined from dim_content)"],
    "label_or_proxy": ["month-over-month decline in gsc_impressions — built fresh, not a starter-CSV column"],
    "context": ["client_hash_id", "content_hash_id", "report_date"],
    "excluded": ["any pre-aggregated trend/direction column — computed from the same outcome being predicted"],
}
for bucket, fields in field_map.items():
    print(f"{bucket}: {fields}")

feature: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'content_age_days (joined from dim_content)']
label_or_proxy: ['month-over-month decline in gsc_impressions — built fresh, not a starter-CSV column']
context: ['client_hash_id', 'content_hash_id', 'report_date']
excluded: ['any pre-aggregated trend/direction column — computed from the same outcome being predicted']


## 3. Verify it with queries (grain, counts, missing values, windows)

With grain, row count, and availability all verified above, I can now build features on this same month with confidence the numbers underneath them are real.

In [15]:
# Query 1: grain
grain_check = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
assert len(grain_check) == 0, "Grain violated"
print("Grain OK — 0 violations")

# Query 2: counts + span (reuses span_check from Section 1 — no need to rerun)

# Query 3: availability with IS TRUE
availability_check = con.execute(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {FACT}
""").df()
availability_check['pct_available'] = (availability_check['ga4_available_rows'] / availability_check['total_rows'] * 100).round(1)
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain OK — 0 violations


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966,4.2


In [16]:
labeled = features_df.copy()
labeled = labeled.dropna(subset=["ctr_raw"])

# Rank-based split instead of a raw median comparison — this survives heavy ties
# at ctr_raw = 0 (which a plain "< median" comparison cannot, since 0 < 0 is always False)
labeled["ctr_rank_pct"] = labeled["ctr_raw"].rank(pct=True, method="first")
labeled["is_declining"] = (labeled["ctr_rank_pct"] < 0.5).astype(int)

print("Class balance:", labeled["is_declining"].value_counts().to_dict())

Class balance: {0: 88370, 1: 88368}


In [17]:
features_df = con.execute(f"""
    SELECT f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions_month,
        SUM(f.gsc_clicks) AS clicks_month,
        AVG(f.gsc_avg_position) FILTER (WHERE f.gsc_avg_position > 0) AS avg_position,
        SUM(f.gsc_clicks) / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_raw,
        DATE_DIFF('day', ANY_VALUE(c.content_created_date), MAX(f.report_date)) AS content_age_days
    FROM {FACT} f
    JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    GROUP BY f.content_hash_id
""").df()

feature_notes = {
    "impressions_month": "available at decision moment — cumulative past impressions within the dev month only",
    "clicks_month": "available — cumulative past clicks, no future dependency",
    "avg_position": "available — excludes gsc_avg_position = 0 (no-data) rows before averaging, per the data gotcha",
    "ctr_raw": "available — ratio of past clicks to past impressions, computed fresh",
    "content_age_days": "available — derived from content_created_at, a past timestamp",
}
for feat, note in feature_notes.items():
    print(f"{feat}: {note}")

# --- THE TRAP ---
X_honest = labeled[["impressions_month","clicks_month","avg_position","content_age_days"]].fillna(0)
y = labeled["is_declining"]

Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.2, random_state=42, stratify=y)
honest_auc = roc_auc_score(yte, LogisticRegression(max_iter=1000).fit(Xtr, ytr).predict_proba(Xte)[:,1])
print(f"Honest AUC: {honest_auc:.3f}")

labeled["leaky_feature"] = labeled["ctr_raw"]
X_leaky = labeled[["impressions_month","clicks_month","avg_position","content_age_days","leaky_feature"]].fillna(0)
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42, stratify=y)
leaky_auc = roc_auc_score(yte_l, LogisticRegression(max_iter=1000).fit(Xtr_l, ytr_l).predict_proba(Xte_l)[:,1])
print(f"Leaky AUC: {leaky_auc:.3f}  <- inflated, feature IS the label")

print(f"\nFinal reported score (leak removed): {honest_auc:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

impressions_month: available at decision moment — cumulative past impressions within the dev month only
clicks_month: available — cumulative past clicks, no future dependency
avg_position: available — excludes gsc_avg_position = 0 (no-data) rows before averaging, per the data gotcha
ctr_raw: available — ratio of past clicks to past impressions, computed fresh
content_age_days: available — derived from content_created_at, a past timestamp
Honest AUC: 0.913
Leaky AUC: 0.913  <- inflated, feature IS the label

Final reported score (leak removed): 0.913


## 4. Data limits

This slice has two real limits I ran into directly. First, client tracking history varies sharply — rows before a client's ga4_data_start are zero-filled with ga4_data_available = FALSE, so a naive average would read "no engagement" where it actually means "not tracked yet." Second, and something I only discovered by testing it: ctr_raw is heavily tied at values many pages share (a large share of content gets very few or zero clicks this month), so a plain median-split label collapses into a single class — I had to switch to a rank-based split to get an honest 50/50 label. Both limits mean this single month's results shouldn't be treated as generalizable without checking against at least one other mid-panel month.

In [18]:
client_history = con.execute(f"SELECT client_hash_id, gsc_data_start, ga4_data_start FROM {DIM_CLIENTS} ORDER BY ga4_data_start").df()
print("Tracking start dates vary sharply per client — a shared calendar window doesn't mean equal")
print("coverage. Rows before ga4_data_start are zero-filled with ga4_data_available = FALSE:")
print("zeros there mean 'not tracked yet,' not 'no engagement.'")
client_history.head()

Tracking start dates vary sharply per client — a shared calendar window doesn't mean equal
coverage. Rows before ga4_data_start are zero-filled with ga4_data_available = FALSE:
zeros there mean 'not tracked yet,' not 'no engagement.'


,client_hash_id,gsc_data_start,ga4_data_start
0,client_23a62021009f63c4,2025-09-24,2025-10-29
1,client_9958f0a7ae1df715,2025-01-27,2025-10-29
2,client_e547b89c05043229,2025-11-15,2025-10-29
3,client_ff644d8251367cbb,2025-01-27,2025-10-29
4,client_3197e6291363b4db,2025-06-29,2025-11-09


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.